# CRAFT

In [1]:
import torch, torchvision, cv2, skimage, scipy

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("opencv:", cv2.__version__)
print("scikit-image:", skimage.__version__)
print("scipy:", scipy.__version__)

torch: 2.6.0+cu124
torchvision: 0.21.0+cu124
opencv: 4.10.0
scikit-image: 0.25.2
scipy: 1.11.4


In [3]:
import numpy as np
np.__version__

'1.26.4'

In [2]:
import torch
from craft import CRAFT
from utils import imgproc
from craft_utils import getDetBoxes
import cv2
import numpy as np
import os

# Set path to model and image
image_path = "../../dataset/basket/negative_005.jpg"
model_path = "../../papers/craft/craft_mlt_25k.pth"

# Load CRAFT model
net = CRAFT()
net.load_state_dict(torch.load(model_path, map_location='cpu'))
net.eval()

# Preprocess image
image = imgproc.loadImage(image_path)
img_resized, target_ratio, size_heatmap = imgproc.resize_aspect_ratio(image, 1280, interpolation=cv2.INTER_LINEAR)
ratio_h = ratio_w = 1 / target_ratio

x = imgproc.normalizeMeanVariance(img_resized)
x = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0)  # [B, C, H, W]

# Forward pass
with torch.no_grad():
    y, _ = net(x)

# Get text boxes
score_text = y[0, :, :, 0].cpu().data.numpy()
boxes, _ = getDetBoxes(score_text, text_threshold=0.7, link_threshold=0.4, low_text=0.4)

# Post-process boxes to crop image regions
for i, box in enumerate(boxes):
    box = np.array(box).astype(np.int32).reshape((-1, 1, 2))
    rect = cv2.boundingRect(box)
    x, y, w, h = rect
    cropped = image[y:y+h, x:x+w]
    cv2.imwrite(f"text_region_{i}.png", cropped)

print("✅ Cropped text regions saved.")

ModuleNotFoundError: No module named 'craft'

# PaddleOCR

In [7]:
import paddleocr
print(dir(paddleocr))
print(paddleocr.__version__)

['DocImgOrientationClassification', 'DocPreprocessor', 'DocUnderstanding', 'DocVLM', 'FormulaRecognition', 'FormulaRecognitionPipeline', 'LayoutDetection', 'PPChatOCRv4Doc', 'PPDocTranslation', 'PPStructureV3', 'PaddleOCR', 'SealRecognition', 'SealTextDetection', 'TableCellsDetection', 'TableClassification', 'TableRecognitionPipelineV2', 'TableStructureRecognition', 'TextDetection', 'TextImageUnwarping', 'TextLineOrientationClassification', 'TextRecognition', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_abstract', '_common_args', '_constants', '_env', '_models', '_pipelines', '_utils', '_version', 'logger']
3.1.0


In [9]:
img_path = '../../dataset/basket/negative_005.jpg'

# Initialize PaddleOCR instance
from paddleocr import PaddleOCR
ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False)

# Run OCR inference on a sample image 
result = ocr.predict(
    input=img_path)

# Visualize the results and save the JSON results
for res in result:
    res.print()
    res.save_to_img("output")
    res.save_to_json("output")

/home/soham/.local/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', None)
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in /home/soham/.paddlex/official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

inference.yml:   0%|          | 0.00/903 [00:00<?, ?B/s]

inference.pdiparams:   0%|          | 0.00/87.9M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

inference.pdiparams:   0%|          | 0.00/84.4M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]


KeyboardInterrupt



In [1]:
# /home/soham/Documents/ISI_RI/dataset/Dataset

In [2]:
# !pip install pytesseract pillow --break-system-packages

In [3]:
import os
from PIL import Image
from collections import defaultdict, Counter

import pytesseract

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

In [4]:
# Optional: If tesseract is not in PATH
# pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'  # Check with `which tesseract`

# Path to your image
# image_path = "./Med/Train/negative/negative11.jpg"

# image_path = "./Med/Train/negative/negative5.jpeg"
image_path = "../../dataset/basket"

Extracted Text:



In [5]:
# Define standard aspect ratios
standard_ratios = {
    "1:1": 1.0, # 1 BOX
    "4:3": 4/3, # 1.33 Horizontal
    "3:2": 3/2, # 1.5 Horizontal
    "16:9": 16/9, # 1.78 Horizontal
    "5:4": 5/4, # 1.25 Horizontal
    "21:9": 21/9, # 2.33 Horizontal
    "2:1": 2.0, # 2 Horizontal
    "7:5": 7/5, # 1.4 Horizontal
    # "A4 (1.41:1)": 1.41,
    "3:4": 3/4, # 0.75 Vertical
    "2:3": 2/3, # 0.67 Vertical
    "9:16": 9/16, # 0.5625 Vertical
    "5:7": 5/7, # 0.71 Vertical
    "4:5": 4/5, # 0.8 Vertical
    "A-series (1:√2)": 1/1.414, # 0.707 Vertical
    "2:5": 2/5, # 0.4 Vertical
    "Other":0
}

# Function to match image to closest standard aspect ratio
def closest_standard_ratio(width, height, tolerance=0.05):
    if height == 0:
        return "Invalid"
    ratio = width / height
    for name, std_ratio in standard_ratios.items():
        if abs(ratio - std_ratio) <= tolerance:
            return name
    return "Other"

In [6]:
all_df = dict()
total_df = dict()

# ORIENTATION analysis

In [36]:
aspect_ratio_orientation = {
    # --- Box-like (almost square) ---
    "1:1": "box",

    # --- Horizontal (landscape) ---
    "4:3": "horizontal",      # 1.33
    "3:2": "horizontal",      # 1.5
    "16:9": "horizontal",     # 1.78
    "5:4": "horizontal",      # 1.25
    "21:9": "horizontal",     # 2.33
    "2:1": "horizontal",      # 2.0
    "7:5": "horizontal",      # 1.4
    "A4 (1.41:1)": "horizontal",  # 1.41

    # --- Vertical (portrait) ---
    "3:4": "vertical",        # 0.75
    "2:3": "vertical",        # 0.666
    "9:16": "vertical",       # 0.5625
    "5:7": "vertical",        # 0.714
    "4:5": "vertical",        # 0.8
    "A-series (1:√2)": "vertical",  # ≈0.707
    "2:5": "vertical",        # 0.4

    # --- Catch-all ---
    # "Other": "other"
    "Other": "box"
}


columnNames = ["vertical", "horizontal","box"]
rowNames = ["positive", "negative", "neutral"]

# Create a DataFrame filled with zeros
orientation_df = pd.DataFrame(0, index=rowNames, columns=columnNames)

In [37]:
total_df

,1:1,4:3,3:2,16:9,5:4,21:9,2:1,7:5,3:4,2:3,9:16,5:7,4:5,A-series (1:√2),2:5,Other
positive,259,35,70,96,13,0,11,17,81,83,4,0,202,0,0,150
negative,82,112,55,433,20,0,12,13,33,10,0,0,10,0,0,127
neutral,199,49,41,273,35,5,12,4,55,29,12,0,41,0,2,190


In [38]:
orientation_df

,vertical,horizontal,box
positive,0,0,0
negative,0,0,0
neutral,0,0,0


In [40]:
# positive - 1:1, Other
# negative - 16:9, Other
# netural - 1:1, 16:9
# seems like 'neutral' is a combination of 'positive' and 'negative'

In [41]:
for aspect_ratio in total_df.keys():
    orientation_df[aspect_ratio_orientation[aspect_ratio]]+=total_df[aspect_ratio]

In [42]:
orientation_df

,vertical,horizontal,box
positive,370,242,409
negative,53,645,209
neutral,139,419,389


In [43]:
# Convert to matrix
matrix = orientation_df.values  # shape: (3, 3)

In [44]:
orientation_dist = matrix.astype(float).copy()
# print(orientation_dist)
# Orientation distribution per sentiment
l = len(matrix)
for i in range(l):
    value = matrix[i]/np.sum(matrix[i])
    orientation_dist[i] = matrix[i]/np.sum(matrix[i])
    # print(orientation_dist[i])


sentiment_dist = matrix.T.astype(float).copy()
transp_mat = matrix.T
l = len(matrix)
for i in range(l):
    value = transp_mat[i]/np.sum(transp_mat[i])
    sentiment_dist[i] = transp_mat[i]/np.sum(transp_mat[i])

sentiment_dist = np.round(sentiment_dist, 2)
orientation_dist = np.round(orientation_dist, 2)


orientation_dist_df = pd.DataFrame(orientation_dist, index=rowNames, columns=columnNames)
sentiment_dist_df = pd.DataFrame(sentiment_dist.T, index=rowNames, columns=columnNames)

In [45]:
# matrix[0,1]
# type(matrix)
# matrix.T

In [46]:
orientation_dist_df

,vertical,horizontal,box
positive,0.36,0.24,0.40
negative,0.06,0.71,0.23
neutral,0.15,0.44,0.41


In [47]:
sentiment_dist_df

,vertical,horizontal,box
positive,0.66,0.19,0.41
negative,0.09,0.49,0.21
neutral,0.25,0.32,0.39


# Hypothesis Testing

In [48]:
orientation_df

,vertical,horizontal,box
positive,370,242,409
negative,53,645,209
neutral,139,419,389


In [49]:
import numpy as np
import pandas as pd

# Contingency table
# data = np.array([
#     [370, 242, 409],  # Positive
#     [53, 645, 209],   # Negative
#     [139, 419, 389]   # Neutral
# ])

data = matrix.copy()

# Optional: Set labels
sentiments = ['positive', 'negative', 'neutral']
orientations = ['vertical', 'horizontal', 'box']

# df = pd.DataFrame(data, index=sentiments, columns=orientations)
# Contingency table

# Chi-square test
chi2, p, dof, expected = chi2_contingency(orientation_df)

print(f"Chi-square statistic = {chi2:.3f}")
print(f"p-value = {p}")
print(f"Degrees of freedom = {dof}")

Chi-square statistic = 536.719
p-value = 7.642531909693983e-115
Degrees of freedom = 4


In [50]:
print("Expected frequencies:")
pd.DataFrame(np.round(expected, 2), index=sentiments, columns=orientations)

Expected frequencies:


,vertical,horizontal,box
positive,199.58,463.80,357.62
negative,177.30,412.01,317.69
neutral,185.12,430.19,331.70


In [51]:
# type(expected)

In [52]:
p

7.642531909693983e-115

In [53]:
# Decision
alpha = 0.05
if p < alpha:
    print("Reject the null hypothesis: There is a relationship between orientation and sentiment.")
else:
    print("Fail to reject the null hypothesis: No significant relationship found.")

Reject the null hypothesis: There is a relationship between orientation and sentiment.


In [54]:
import statsmodels.api as sm

# Count of positive vertical
count = 370
# Total vertical images
nobs = orientation_df['vertical'].sum()

ci_low, ci_upp = sm.stats.proportion_confint(count, nobs, alpha=0.05, method='wilson')
print(f"95% CI for P(Positive | Vertical): ({ci_low:.3f}, {ci_upp:.3f})")

95% CI for P(Positive | Vertical): (0.618, 0.696)
